# 05 Analyze and Export Results

Objective: export the compact table, one figure, and a short observation template for the IST manuscript.


In [ ]:
from pathlib import Path
import os
import sys
import importlib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
while not (PROJECT_ROOT / "AGENTS.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import eval_utils as eu
eu = importlib.reload(eu)

CONFIG_PATH = PROJECT_ROOT / "config.json"
if not CONFIG_PATH.exists():
    CONFIG_PATH = PROJECT_ROOT / "config.example.json"
CONFIG = eu.load_config(CONFIG_PATH)
eu.ensure_project_dirs(PROJECT_ROOT)
BENCHMARK_VARIANT = os.getenv("BENCHMARK_VARIANT", "must").strip().lower()
VARIANT_SUFFIX = eu.variant_suffix(BENCHMARK_VARIANT)

PROJECT_ROOT, CONFIG_PATH, BENCHMARK_VARIANT


## Load Scores and Summaries


In [ ]:
benchmark_path = eu.variant_path(PROJECT_ROOT / "data/processed/benchmark_items.csv", BENCHMARK_VARIANT)
scores_path = eu.variant_path(PROJECT_ROOT / "data/processed/uq_scores.csv", BENCHMARK_VARIANT)
summary_path = eu.variant_path(PROJECT_ROOT / "data/processed/metrics_summary.csv", BENCHMARK_VARIANT)
ci_path = eu.variant_path(PROJECT_ROOT / "data/processed/bootstrap_seed_ci.csv", BENCHMARK_VARIANT)

benchmark = eu.read_csv_rows(benchmark_path)
scores = eu.read_csv_rows(scores_path)
summary = eu.read_csv_rows(summary_path)
ci_rows = eu.read_csv_rows(ci_path)
print(f"Benchmark variant: {BENCHMARK_VARIANT}")
print(f"Scores: {len(scores)}")
print(f"Summary rows: {len(summary)}")


## Export Paper-Facing Table


In [ ]:
paper_fields = [
    "model",
    "task",
    "uq_method",
    "accuracy",
    "f1_or_macro_f1",
    "over_commitment",
    "brier",
    "ece",
    "auroc",
    "error_detection_auroc",
    "monotonicity_violations",
    "high_conf_overcommit_80",
    "high_conf_overcommit_90",
    "text_modality_accuracy",
    "label_text_consistency",
    "text_over_commitment",
    "text_high_conf_overcommit_90",
]
table_md = eu.markdown_table(summary, paper_fields)
table_path = eu.variant_path(PROJECT_ROOT / "outputs/paper_results_table.md", BENCHMARK_VARIANT)
table_path.write_text(table_md + "\n", encoding="utf-8")
print(f"Wrote table: {table_path}")
print(table_md)


## Export Compact Figure


In [ ]:
figure_path = eu.variant_path(PROJECT_ROOT / "outputs/task1_p_yes_by_modality.svg", BENCHMARK_VARIANT)
eu.write_task1_modality_svg(scores, figure_path)
print(f"Wrote figure: {figure_path}")


## Export Qualitative Over-Commitment Examples


In [ ]:
example_paths = eu.write_qualitative_overcommitment_examples(
    scores,
    benchmark,
    PROJECT_ROOT / "outputs",
    suffix=VARIANT_SUFFIX,
    limit=5,
    threshold=0.80,
)
print(f"Wrote qualitative examples CSV: {example_paths['csv']}")
print(f"Wrote qualitative examples Markdown: {example_paths['markdown']}")


## Export UQ Method Inventory


In [ ]:
inventory_paths = eu.write_uq_method_inventory(PROJECT_ROOT / "outputs", suffix=VARIANT_SUFFIX)
print(f"Wrote UQ inventory Markdown: {inventory_paths['markdown']}")
print(f"Wrote UQ inventory CSV: {inventory_paths['csv']}")


## Export Manuscript Observation Notes


In [ ]:
notes = [
    "# Result Notes for IST Manuscript",
    "",
    "Fill this file after inspecting the metric table and figure.",
    "",
    "## Observations",
    "- Observation: <grounded result from metrics_summary.csv>.",
    "- Observation: <grounded result from task1_p_yes_by_modality.svg>.",
    "- Observation: <grounded high-confidence over-commitment result>.",
    "",
    "## Interpretation",
    "- Hypothesis: <what the observed pattern may imply>.",
    "",
    "## Caveats",
    "- Controlled variants are synthetic minimal pairs.",
    "- Confidence values are verbalized or consistency-derived, not direct internal model uncertainty.",
    "- Local model IDs and endpoint configuration must be reported exactly.",
    "",
    "## Recommended Next Step",
    "- Recommendation: <best follow-up experiment or paper edit>.",
]
notes_path = eu.variant_path(PROJECT_ROOT / "outputs/result_notes_template.md", BENCHMARK_VARIANT)
notes_path.write_text("\n".join(notes) + "\n", encoding="utf-8")
print(f"Wrote notes template: {notes_path}")
